# Step 7: Prediction - Comparing 2 Models
- Model 1: DistilBERT (ner_model_final)
- Model 2: BiLSTM+CRF Improved (bilstm_crf_improved.keras)

In [8]:
import json, numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
MAX_LEN = 128

with open('D:/Python/NLP/CK/label_mappings.json', 'r') as f:
    mappings = json.load(f)
    id2label = mappings['id2label']
    label2id = mappings['label2id']

import pandas as pd, ast
df = pd.read_csv('D:/Python/NLP/CK/final_data.csv')
df['tokens'] = df['tokens'].apply(ast.literal_eval)
all_words = []
for tokens in df['tokens']: all_words.extend(tokens)
word_counts = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in word_counts.most_common() if c >= 1]
word2id = {w: i for i, w in enumerate(vocab)}
del df
print("Mappings and vocabulary loaded!")

Mappings and vocabulary loaded!


In [9]:
# Load DistilBERT
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

print("Loading DistilBERT model...")
bert_model_path = "D:/Python/NLP/CK/ner_model_final"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_path)
bert_model = AutoModelForTokenClassification.from_pretrained(bert_model_path)
bert_model.eval()
print("DistilBERT loaded!")

Loading DistilBERT model...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2440.47it/s]

DistilBERT loaded!


In [10]:
# Load BiLSTM+CRF
import tensorflow as tf
from tensorflow import keras

print("Loading BiLSTM+CRF model...")
bilstm_model = keras.models.load_model('D:/Python/NLP/CK/bilstm_crf_improved.keras', compile=False)
bilstm_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
print("BiLSTM+CRF loaded!")

Loading BiLSTM+CRF model...
BiLSTM+CRF loaded!


In [11]:
def predict_bilstm(text):
    tokens = text.split()
    token_ids = [word2id.get(w, word2id['<UNK>']) for w in tokens]
    token_ids += [word2id['<PAD>']] * (MAX_LEN - len(token_ids))
    logits = bilstm_model.predict(np.array([token_ids]), verbose=0)[0]
    pred_ids = np.argmax(logits, axis=-1)
    return [(tokens[i], id2label.get(str(pred_ids[i]), 'O')) for i in range(len(tokens))]

In [12]:
# Test predictions
test_sentences = [
    "Joe Biden met Donald Trump in Washington D.C. on Monday",
    "Apple Inc. announced $1 billion investment in New York",
    "The meeting is at the White House on January 15, 2024",
    "Google CEO Sundar Pichai spoke at the conference"
]

print("="*70)
print("PREDICTIONS COMPARISON")
print("="*70)

for text in test_sentences:
    print(f"\nInput: {text}")
    print("-" * 50)
    bert_preds = predict_distilbert(text)
    print("DistilBERT:")
    for word, label in bert_preds:
        print(f"  {word:<15} -> {label}")
    bilstm_preds = predict_bilstm(text)
    print("BiLSTM+CRF:")
    for word, label in bilstm_preds:
        print(f"  {word:<15} -> {label}")

PREDICTIONS COMPARISON

Input: Joe Biden met Donald Trump in Washington D.C. on Monday
--------------------------------------------------
DistilBERT:
  Joe             -> B-PERSON
  Biden           -> I-PERSON
  met             -> B-ORG
  Donald          -> B-PERSON
  Trump           -> I-PERSON
  in              -> O
  Washington      -> B-GPE
  D.C.            -> I-GPE
  on              -> O
  Monday          -> B-DATE
BiLSTM+CRF:
  Joe             -> B-PERSON
  Biden           -> I-PERSON
  met             -> B-ORG
  Donald          -> I-PERSON
  Trump           -> I-PERSON
  in              -> O
  Washington      -> B-GPE
  D.C.            -> I-GPE
  on              -> O
  Monday          -> B-DATE

Input: Apple Inc. announced $1 billion investment in New York
--------------------------------------------------
DistilBERT:
  Apple           -> B-ORG
  Inc.            -> I-ORG
  announced       -> O
  $1              -> B-MONEY
  billion         -> I-MONEY
  investment      -> O
  in

In [13]:
def predict_with_both_models(text, model_choice='both'):
    results = {}
    if model_choice in ['distilbert', 'both']:
        results['distilbert'] = predict_distilbert(text)
    if model_choice in ['bilstm', 'both']:
        results['bilstm'] = predict_bilstm(text)
    return results

# Example
text = "Tim Cook is the CEO of Apple Inc."
predictions = predict_with_both_models(text, 'both')

print(f"Input: {text}\n")
for model_name, preds in predictions.items():
    print(f"{model_name.upper()}:")
    for word, label in preds:
        print(f"  {word} -> {label}")

Input: Tim Cook is the CEO of Apple Inc.

DISTILBERT:
  Tim -> B-PERSON
  Cook -> I-PERSON
  is -> O
  the -> O
  CEO -> O
  of -> O
  Apple -> B-ORG
  Inc. -> I-ORG
BILSTM:
  Tim -> B-PERSON
  Cook -> O
  is -> O
  the -> O
  CEO -> O
  of -> O
  Apple -> B-ORG
  Inc. -> I-ORG


In [14]:
print("="*70)
print("PREDICTION SUMMARY")
print("="*70)
print("""
### Models:
| Model         | Path                      | Parameters  |
|--------------|--------------------------|------------|
| DistilBERT    | ner_model_final/           | 66M       |
| BiLSTM+CRF   | bilstm_crf_improved.keras| ~5M       |

### Usage:
- predict_distilbert(text)
- predict_bilstm(text)
- predict_with_both_models(text, 'both')
""")

PREDICTION SUMMARY

### Models:
| Model         | Path                      | Parameters  |
|--------------|--------------------------|------------|
| DistilBERT    | ner_model_final/           | 66M       |
| BiLSTM+CRF   | bilstm_crf_improved.keras| ~5M       |

### Usage:
- predict_distilbert(text)
- predict_bilstm(text)
- predict_with_both_models(text, 'both')

